In [ ]:
import os
import sys
import subprocess
import numpy as np
from IPython.display import Image, display

Note that the working directory will be set to `lib_dir` (the PROFET code directory), not the notebooks directory. <br>
Libraries such as GPA and Force-Matching are located in `lib_dir`. <br>
So, change the working directory to `os.chdir(lib_dir)` before running GPA or Force-Matching in this notebook.

In [ ]:
lib_dir = os.getcwd()
if "notebooks" in lib_dir:
    lib_dir = os.path.join(lib_dir[:-10], 'PROFET')
elif lib_dir == "/content":
    lib_dir = lib_dir + "/GPA-source_code"
os.chdir(lib_dir)
print(lib_dir)

main_dir = os.path.dirname(lib_dir)
sys.path.insert(0, main_dir)  # for util.utils
sys.path.insert(0, lib_dir)   # for models.velocityfield
data_dir = os.path.join(main_dir, 'data', '')

os.makedirs(os.path.join(main_dir, 'data'), exist_ok=True)
os.makedirs(os.path.join(main_dir, 'assets'), exist_ok=True)

from util.utils import (
    ResourceMonitor, contrast_colors,
    reduce_dimension, save_preprocessed_data, load_preprocessed_data, visualize_data,
    generate_animation, generate_W2distance_plot,
)

## Patient 862 dataset

### Data property

* 2 snapshots at different timepoints (palbociclib treatment, NatMed cohort)
* 115 genes · 17,260 total cells
* Training timepoints: `times=[0, 1]`, `d_red=2`
* Cells per training tp: Day 0: 8,953 · Day 1: 8,307
* Held-out / Intermediate: None (2-timepoint trajectory)

### Data structure
Datasets are located in `data/Patient_862` directory.

* `combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt` : Raw data matrix in $\mathbb{R}^{M \times N}$ composed of $M$ rows for individual samples and $N$ columns of different genes.
* `cell_matrix_palbo_862_1.txt` : Time (day) label for each cell.

In [ ]:
# Preprocessing function
def load_862_dataset():
    import pandas as pd

    # gene expression dataset => full_matrix
    GE_matrix_file = data_dir + 'Patient_862/combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:, 1:]
    else:
        print("Gene expression matrix not found:", GE_matrix_file)

    # time information => time_label
    time_info_file = data_dir + 'Patient_862/cell_matrix_palbo_862_1.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))

    return full_matrix, time_label

In [ ]:
# Data preprocess
example_name = 'Patient_862'
d_reds = [2, 4, 8, 16, 32, 64, 116]  # [] or list of reduced dimensions

full_matrix, time_label = load_862_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_preprocessed_data(data, example_name)

In [ ]:
from collections import Counter
cnt = Counter(time_label)

In [ ]:
# Load precomputed preprocessed data
try:
    time_label, full_matrix, projected_matrix, pca = load_preprocessed_data(example_name)
except:
    time_label, full_matrix = load_preprocessed_data(example_name)

time_points = sorted(list(set(time_label)))

## PROFET

In [ ]:
# PROFET step 1: GPA
d_red = 2
times = [0, 1]       # day 0 (pre-treatment) -> day 1 (post-palbociclib)
intermediate_times = []  # no intermediate snapshots
dimension_reduction = 'Y'
exp_no = "test"
exp_memo = f"dim{d_red}_{exp_no}"

result_dir = os.path.join(main_dir, "assets", example_name, exp_memo)
os.makedirs(result_dir, exist_ok=True)

# tunable parameters
adjust_lr_P_epochs = 'N'    # use fixed lr_P and epochs below
lr_P = 0.0025
epochs = 5000
specify_f_Lip_threshold = 'Y'
f_Lip_threshold = 2e-5

In [ ]:
os.chdir(lib_dir)

dimension_note = f'dim{d_red}-' if dimension_reduction in ['Y', 'yes'] else ''

with ResourceMonitor() as monitor:
    for i in range(len(times) - 1):
        cmd = (
            f"python3 run_GPA.py"
            f" --dataset {example_name}"
            f" --label {times[i]} {times[i+1]}"
            f" --N_dim {d_red}"
            f" --exp_no {exp_no}"
            f" --dimension_reduction {dimension_reduction}"
        )
        if adjust_lr_P_epochs in ['N', 'no']:
            cmd += f" --adjust_lr_P_epochs {adjust_lr_P_epochs} -lr_P {lr_P} --epochs {epochs}"
        if specify_f_Lip_threshold in ['Y', 'yes']:
            cmd += f" --f_Lip_threshold {f_Lip_threshold}"
        subprocess.run(cmd, shell=True, check=True)

monitor.report("PROFET GPA", dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, "resources.txt"))

In [ ]:
# PROFET step 2: Force-Matching
if dimension_reduction in ['Y', 'yes']:
    dimension_note = f'dim{d_red}-'
else:
    dimension_note = ''
gpa_filenames = [
    f'KL-Lipschitz_1.0000-{times[0]}_{times[1]}times-{dimension_note}{cnt[times[1]]:04d}_{cnt[times[0]]:04d}-00-{exp_no}.pickle',
]

gpa_filenames_join = " ".join(gpa_filenames)
times_join = " ".join([str(t) for t in times])

In [ ]:
os.chdir(lib_dir)
cmd = (
    f"python3 run_ForceMatching.py"
    f" --dataset {example_name}"
    f" --ts {times_join}"
    f" --exp_memo {exp_memo}"
    f" --files {gpa_filenames_join}"
)

with ResourceMonitor() as monitor:
    subprocess.run(cmd, shell=True, check=True)

monitor.report("PROFET ForceMatching", dataset=example_name, d_red=d_red,
               filename=os.path.join(result_dir, "resources.txt"))

In [ ]:
from models.velocityfield import VelocityField

# load velocityfield
velocity_net, p = VelocityField.load(os.path.join(main_dir, "assets", example_name, exp_memo, ""))

# integrate ODE
dt = p['numerical_ts'][-1] / 200
numerical_dt = dt
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]
input_data = full_matrix[time_label == times[0], :] if dimension_reduction in ['N', 'no'] else projected_matrix[time_label == times[0], :d_red]
X1_trpts = velocity_net.integrate(input_data, T=p['numerical_ts'][-1], dt=dt)

In [ ]:
img_src1 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-movie-particles.gif")
img_src2 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-movie-velocities.gif")
img_src3 = os.path.join(main_dir, "assets", example_name, f"{exp_memo}-w2distances.png")

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src1, d_red, True, colors=contrast_colors, plot_vectorfield=False)
display(Image(filename=img_src1))

generate_animation(example_name, times, intermediate_times, X1_trpts,
                   img_src2, d_red, True, colors=contrast_colors, plot_vectorfield=True)
display(Image(filename=img_src2))

generate_W2distance_plot(example_name, times, intermediate_times, X1_trpts,
                         img_src3, d_red, True, colors=contrast_colors)
display(Image(filename=img_src3))

## Downstream Analysis — Phenotypic Shift Heterogeneity

Reconstructs the heterogeneity of phenotypic shift for the Patient 862 dataset.

**Prerequisites:**
- PROFET section above must have been run so that `X1_trpts`, `p`, `full_matrix`, `time_label`, `time_points`, and `pca` are in scope.
- The displacement histogram (produced in the first cell below) is used to identify thresholds for Low / Medium / High phenotypic shift classification. Inspect that plot before running the classification cell.

In [ ]:
# Downstream analysis setup
import pandas as pd
from sklearn import decomposition

# d_red-component PCA for gene expression reconstruction via inverse_transform
pca_d = decomposition.PCA(n_components=d_red, random_state=0)
pca_d.fit(full_matrix)

# Per-timepoint full expression matrices
mats = {t: full_matrix[time_label == t].astype(float) for t in time_points}

# Gene names from the expression matrix file
gene_file = data_dir + 'Patient_862/combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt'
df_genes = pd.read_table(gene_file, sep="\t")
gene_names = df_genes.columns[1:].tolist()
print(f"Number of genes: {len(gene_names)}")

# Import downstream analysis functions
from util.downstream import (
    plot_X1_hat_displacement_distribution,
    generate_static_cluster_plot_deviation_colormap_862,
    Average_gene_dynamics_whole_saveonly_single_trajectory_clinical,
)

source_t, target_t = times[0], times[-1]  # 0, 4
index = 1
max_i = 200

### Cell State Displacement Distribution

Computes Euclidean displacement of each cell from pre- to post-treatment in PCA space. Inspect the histogram to choose thresholds for Low / Medium / High phenotypic shift classification in the next cell.

In [ ]:
csv_output   = os.path.join(result_dir, f"{exp_memo}_X1_hat_displacement_stats.csv")
plot_output  = os.path.join(result_dir, f"{exp_memo}_X1_hat_displacement_distribution.png")
hist_output  = os.path.join(result_dir, f"{exp_memo}_X1_hat_displacement_histogram.csv")

plot_X1_hat_displacement_distribution(X1_trpts, csv_output, plot_output, hist_output, exp_memo=exp_memo)
display(Image(filename=plot_output))

### Phenotypic Shift Classification and Visualization

Classifies cells into **Low / Medium / High** phenotypic shift groups based on displacement thresholds read from the histogram above, and generates trajectory plots coloured by class. Cluster labels are saved to `{exp_memo}_X1_hat_deviation.csv` for use in subsequent cells.

Thresholds are dataset-specific (hardcoded inside `generate_static_cluster_plot_deviation_colormap_862`):
```
low    : displacement ≤ 1.288
medium : 1.288 < displacement ≤ 2.365
high   : displacement > 2.365
```

In [ ]:
cluster_output_file = os.path.join(result_dir, f"{exp_memo}_static_celltypes_plot_deviation_colormap.png")

X1_hat_labels = generate_static_cluster_plot_deviation_colormap_862(
    pca_d, source_t, target_t, start_i=0, X1_trpts=X1_trpts,
    mats=mats, index=index, intermediate_t=[], output_file=cluster_output_file,
)
display(Image(filename=cluster_output_file))

# Save cluster labels
cluster_save_path = os.path.join(result_dir, f"{exp_memo}_X1_hat_deviation.csv")
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),
    "Cluster_Label": X1_hat_labels,
})
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")

### Single-Cell Gene Expression Dynamics

For each gene, plots average gene expression trajectories split by Low / Medium / High phenotypic shift subgroup. Output PNGs are saved to `result_dir`.

In [ ]:
for gene in gene_names:
    print(f"Processing gene: {gene}")
    try:
        subgroup_output_file = os.path.join(result_dir, f"{exp_memo}_single_cell_trajectories_{gene}.png")
        Average_gene_dynamics_whole_saveonly_single_trajectory_clinical(
            pca_d, gene_names, source_t, target_t, X1_trpts, mats,
            gene, index, p, max_i,
            intermediate_t=[target_t],
            subgroup_output_file=subgroup_output_file,
            cluster_save_path=cluster_save_path,
        )
    except Exception as e:
        print(f"  Error: {e}")

## Downstream Analysis — Trajectory Visualization and Subtrajectory Classification

Visualizes full reconstructed trajectories and classifies them into subtrajectories based on **fate cell state** (target clustering) for the Patient 862 dataset.

**Prerequisites:**
- PROFET section above must have been run so that `X1_trpts`, `p`, `physical_dt`, `full_matrix`, `time_label`, `time_points`, and `pca_d` are in scope (the Phenotypic Shift section above defines `pca_d` and `mats`).

In [ ]:
from util.downstream import (
    generate_static_trajectory_plots_two_timepoints_no_middle,
    classify_X1_hat,
    generate_static_cluster_plot_target,
)

# pca_d, mats, source_t, target_t already defined in the Phenotypic Shift section above
optimal_k = 2   # number of subtrajectory classes; adjust based on biology
start_i = 0
index = 10

### Static Trajectory Visualization

Generates static scatter plots of the full reconstructed trajectory: one with trajectory snapshots (coloured by time via viridis gradient) and one showing only the training data snapshots.

In [ ]:
output_with = os.path.join(result_dir, f"{exp_memo}_static_trajectory_with_snapshots.png")
output_without = os.path.join(result_dir, f"{exp_memo}_static_trajectory_without_snapshots.png")

generate_static_trajectory_plots_two_timepoints_no_middle(
    pca=pca_d,
    physical_dt=physical_dt,
    days=times,
    intermediate_days=[],
    X1_trpts=X1_trpts,
    mats=mats,
    d_red=d_red,
    output_file_with_snapshots=output_with,
    output_file_without_snapshots=output_without,
)
display(Image(filename=output_with))
display(Image(filename=output_without))

### Subtrajectory Classification by Fate State

Clusters cells at the **target** time point into `optimal_k` groups using K-means, then assigns each trajectory to the fate cluster it converges toward. Produces an animated GIF of trajectories coloured by fate group and a static initial-state figure.

In [ ]:
output_gif = os.path.join(result_dir, f"{exp_memo}_classify_x1_hat_trajectory.gif")
output_png = os.path.join(result_dir, f"{exp_memo}_fate_state_with_background_x1.png")

classify_X1_hat(
    full_matrix, pca_d, source_t, target_t, X1_trpts, mats,
    optimal_k, start_i, index, p,
    reverse=False, intermediate_t=[],
    d_red=d_red, random_state=42, exp_memo=exp_memo,
    output_file=output_gif, output_file_2=output_png,
)
display(Image(filename=output_png))

### Static Subtrajectory Plot (by Fate State)

Generates a static plot of all trajectory snapshots coloured by fate subgroup. Cluster labels are saved to `{exp_memo}_X1_hat_clusters.csv`.

In [ ]:
cluster_output_file = os.path.join(result_dir, f"{exp_memo}_static_cluster_plot_target.png")

X1_hat_labels = generate_static_cluster_plot_target(
    pca_d, source_t, target_t, X1_trpts, mats,
    optimal_k, start_i, index=1, p=p,
    reverse=False, intermediate_t=[],
    d_red=d_red, random_state=42, exp_memo=exp_memo,
    output_file=cluster_output_file,
)
display(Image(filename=cluster_output_file))

# Save cluster labels
cluster_save_path = os.path.join(result_dir, f"{exp_memo}_X1_hat_clusters.csv")
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),
    "Cluster_Label": X1_hat_labels,
})
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")